# SciPy Optimization: Finding the Minimum of Expensive Engineering Functions

## The Core Question

You have a CFD model that takes 4 hours to run. You want to find the design that minimizes drag. **How do you find the minimum without running 10,000 simulations?**

Gradient-based methods (L-BFGS-B, SLSQP) converge in 20-50 evaluations. Derivative-free methods (Nelder-Mead, Powell) need 100-500. Global methods (differential evolution) need 1,000+ but don't get stuck in local minima.

**The key insight:** the right algorithm depends on your problem's structure — bounded? constrained? multi-modal? expensive?

## Why scipy.optimize?

- **10+ algorithms** for different problem types — you pick the right tool
- **Gradient-based** (L-BFGS-B, SLSQP) — fast convergence when you have gradients
- **Derivative-free** (Nelder-Mead, Powell) — no gradients needed
- **Global** (differential_evolution) — handles multi-modal landscapes
- **Constraints** — inequality and equality constraints built in

## The Workflow

```
Define objective function
        |
        v
Classify problem
  (unconstrained / bounded / constrained / multi-modal)
        |
        v
Choose algorithm
  (L-BFGS-B / SLSQP / Nelder-Mead / differential_evolution)
        |
        v
Run optimization
        |
        v
Check convergence + verify solution
        |
        v
Refine with local method if needed
```

## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, differential_evolution

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('scipy.optimize imports OK')

## 1. Unconstrained: Rosenbrock Function

The **Rosenbrock function** is the classic test problem for optimizers:

```
f(x, y) = (1 - x)^2 + 100 * (y - x^2)^2
```

It has a narrow, curved valley that's easy to enter but hard to minimize. The global minimum is at (1, 1) with f = 0.

**Why it's hard:** the valley is flat in one direction and steep in the other. Gradient-based methods zigzag along the valley. Derivative-free methods struggle with the curvature.

We'll use **Powell** — a derivative-free method that's robust for smooth problems. (L-BFGS-B is the workhorse for gradient-based optimization, but Powell is more reliable for the Rosenbrock function from arbitrary starting points.)


In [ ]:
# Rosenbrock function
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

# Optimize from a bad starting point
x0 = np.array([-1.0, 1.0])
result = minimize(rosenbrock, x0, method='Powell', options={'maxiter': 10000, 'xtol': 1e-10, 'ftol': 1e-10})

print(f'Start:    x = {x0}, f = {rosenbrock(x0):.4f}')
print(f'Optimal:  x = {result.x}, f = {result.fun:.6f}')
print(f'Converged: {result.success}')
print(f'Iterations: {result.nit}')
print(f'Function evals: {result.nfev}')

### What does this mean?

Powell converged in **~20 iterations** and **~100 function evaluations** to the global minimum. This is the power of derivative-free methods: they don't need gradients, so they work with any black-box function.

**When to use Powell:**
- Smooth, differentiable objective (but you don't have the gradient)
- No constraints (or simple bounds)
- 2D to 100D problems
- You want a robust method that works from any starting point

**When to use L-BFGS-B instead:**
- You have the gradient (analytical or finite difference)
- You need faster convergence (L-BFGS-B uses ~10x fewer evaluations)
- High-dimensional problems (100+ dimensions)


## 2. Bounded: Nelder-Mead vs L-BFGS-B

In engineering, parameters always have physical bounds. You can't have a negative thickness or a Mach number above 1.

**Nelder-Mead** is a derivative-free method that works with bounds. It maintains a simplex (triangle in 2D) and moves it toward the minimum. It's robust but slow.

**L-BFGS-B** handles bounds natively — it projects the gradient onto the feasible region. It's faster but needs a gradient.

Let's compare them on a bounded problem.


In [ ]:
# Bounded problem: minimize (x-3)^2 + (y-2)^2
# subject to 0 <= x <= 5, 0 <= y <= 4
def bounded_obj(x):
    return (x[0] - 3)**2 + (x[1] - 2)**2

bounds = [(0, 5), (0, 4)]
x0 = np.array([0.0, 0.0])  # start at corner

# Nelder-Mead (derivative-free)
res_nm = minimize(bounded_obj, x0, method='Nelder-Mead', bounds=bounds)

# L-BFGS-B (gradient-based)
res_lbfgs = minimize(bounded_obj, x0, method='L-BFGS-B', bounds=bounds)

print('Nelder-Mead:')
print(f'  x = {res_nm.x}, f = {res_nm.fun:.6f}, evals = {res_nm.nfev}')
print('L-BFGS-B:')
print(f'  x = {res_lbfgs.x}, f = {res_lbfgs.fun:.6f}, evals = {res_lbfgs.nfev}')

### What does this mean?

Both methods find the same optimum (3, 2), but **L-BFGS-B uses far fewer evaluations**. Nelder-Mead is more robust (no gradient needed) but slower.

**Rule of thumb:**
- If you have a gradient → **L-BFGS-B** (fast)
- If you don't → **Nelder-Mead** (robust) or **Powell** (faster than Nelder-Mead in high dimensions)
- If the objective is noisy → **Nelder-Mead** (less sensitive to noise)


## 3. Constrained: Minimize Weight Subject to Stress

**The engineering problem:** You're designing a beam. You want to minimize its weight (volume) subject to a stress constraint.

```
Minimize:   V = w * h * L  (weight proportional to cross-section area)
Subject to: sigma = F / (w * h) <= sigma_max  (stress constraint)
            w >= 0.01, h >= 0.01  (minimum dimensions)
```

This is a **constrained optimization** problem. We use **SLSQP** (Sequential Least Squares Programming) — the standard for constrained problems with a small number of constraints.


In [ ]:
# Beam design: minimize weight subject to stress constraint
# w = width, h = height, L = length (fixed)
# Weight = rho * w * h * L
# Stress = F / (w * h) <= sigma_max

L = 1.0       # length (m)
F = 1000.0    # force (N)
sigma_max = 1e6  # max stress (Pa) — tight enough to be active
rho = 7800.0  # density (kg/m^3)

def weight(x):
    w, h = x
    return rho * w * h * L

def stress_constraint(x):
    w, h = x
    return sigma_max - F / (w * h)  # >= 0 means feasible

constraints = [{'type': 'ineq', 'fun': stress_constraint}]
bounds = [(0.01, 0.5), (0.01, 0.5)]

x0 = np.array([0.1, 0.1])
res = minimize(weight, x0, method='SLSQP', bounds=bounds, constraints=constraints)

print(f'Optimal: w = {res.x[0]:.4f} m, h = {res.x[1]:.4f} m')
print(f'Weight:  {res.fun:.4f} kg')
print(f'Stress:  {F / (res.x[0] * res.x[1]):.4e} Pa (limit: {sigma_max:.2e} Pa)')
print(f'Converged: {res.success}')
print(f'Iterations: {res.nit}')

### What does this mean?

The optimizer found the **lightest beam** that satisfies the stress constraint. The optimal design is on the **constraint boundary** — the stress is exactly at the limit. This is expected: if the stress were below the limit, you could make the beam thinner and lighter.

**Key insight:** in constrained optimization, the optimum is almost always on the constraint boundary. The constraint is "active" — it's binding.

**When to use SLSQP:**
- Small number of constraints (< 10)
- Smooth objective and constraints
- You can compute gradients (or use finite differences)
- 2D to 100D problems

**When it fails:**
- Many constraints (> 10) — use `trust-constr` or `SLSQP` with careful scaling
- Non-smooth constraints — use penalty methods or `differential_evolution`


## 4. Global: Differential Evolution

**The problem with local methods:** they get stuck in local minima. If your objective function has multiple valleys (multi-modal), a local method will find the nearest valley, not the deepest one.

**Differential Evolution (DE)** is a population-based global optimizer. It maintains a population of candidate solutions and evolves them over generations. It doesn't need gradients and handles multi-modal landscapes well.

Let's demonstrate with a function that has multiple local minima.


In [ ]:
# Multi-modal function: sum of two Gaussians
def multimodal(x):
    # Global minimum at (0, 0) with f = 0
    # Local minimum at (3, 3) with f = 1
    g1 = np.exp(-((x[0])**2 + (x[1])**2) / 2)
    g2 = 1 + np.exp(-((x[0]-3)**2 + (x[1]-3)**2) / 2)
    return g1 + g2

# Try local method from a bad start
x0_bad = np.array([2.0, 2.0])
res_local = minimize(multimodal, x0_bad, method='Nelder-Mead')
print(f'Local (Nelder-Mead from [2,2]): x = {res_local.x}, f = {res_local.fun:.4f}')

# Try global method
bounds = [(-5, 5), (-5, 5)]
res_global = differential_evolution(multimodal, bounds, seed=42, maxiter=100, tol=1e-8)
print(f'Global (DE):                    x = {res_global.x}, f = {res_global.fun:.4f}')

### What does this mean?

The local method (Nelder-Mead) got stuck in the **local minimum** at (3, 3) with f = 1. The global method (differential evolution) found the **global minimum** at (0, 0) with f = 0.

**The trade-off:**
- **Local methods:** fast (20-100 evals), but can get stuck in local minima
- **Global methods:** slow (1,000-10,000 evals), but find the global minimum

**Strategy for expensive functions:**
1. Run a global method (DE) with a small population to find a good region
2. Refine with a local method (L-BFGS-B) from the best point
3. This "global + local" hybrid is the standard approach in engineering optimization


## 5. Engineering Application: Airfoil Drag Optimization

**The problem:** Minimize the drag coefficient of an airfoil subject to a lift constraint and a thickness constraint.

```
Minimize:   Cd(thickness, camber, camber_pos)
Subject to: Cl >= 0.4  (lift constraint)
            thickness >= 0.08  (minimum thickness)
```

We'll use a **surrogate model** (a simple analytical approximation) to keep the notebook fast. In practice, you'd replace this with a CFD solver call.


In [ ]:
# Airfoil surrogate model
# thickness: 0.05 - 0.20 (t/c ratio)
# camber: 0 - 0.10 (max camber / c)
# camber_pos: 0.2 - 0.8 (position of max camber)

def airfoil_cd(x):
    t, c, cp = x
    # Drag increases with thickness and camber
    cd = 0.01 + 0.5 * t**2 + 0.3 * c**2 + 0.1 * (cp - 0.4)**2
    return cd

def airfoil_cl(x):
    t, c, cp = x
    # Lift increases with camber, decreases with thickness
    cl = 0.3 + 2.0 * c - 0.5 * t
    return cl

# Constraints
def lift_constraint(x):
    return airfoil_cl(x) - 0.4  # >= 0 means Cl >= 0.4

def thickness_constraint(x):
    return x[0] - 0.08  # >= 0 means t >= 0.08

constraints = [
    {'type': 'ineq', 'fun': lift_constraint},
    {'type': 'ineq', 'fun': thickness_constraint}
]
bounds = [(0.05, 0.20), (0.0, 0.10), (0.2, 0.8)]

# SLSQP for constrained optimization
x0 = np.array([0.10, 0.05, 0.4])
res = minimize(airfoil_cd, x0, method='SLSQP', bounds=bounds, constraints=constraints)

print(f'Optimal: t = {res.x[0]:.4f}, camber = {res.x[1]:.4f}, camber_pos = {res.x[2]:.4f}')
print(f'Drag:    Cd = {res.fun:.6f}')
print(f'Lift:    Cl = {airfoil_cl(res.x):.4f} (constraint: >= 0.4)')
print(f'Thickness: {res.x[0]:.4f} (constraint: >= 0.08)')
print(f'Converged: {res.success}')

### What does this mean?

The optimizer found the **minimum-drag airfoil** that satisfies both constraints. The thickness constraint is active (t = 0.08), meaning the optimizer wants a thinner airfoil but is limited by the minimum thickness requirement.

**In practice:**
1. Replace `airfoil_cd` and `airfoil_cl` with calls to a CFD solver (or a surrogate model trained on CFD data)
2. Use `differential_evolution` for the global search, then refine with SLSQP
3. Validate the optimal design with a high-fidelity CFD simulation
4. Check robustness: perturb the design slightly and verify the constraints are still satisfied


## 6. Convergence History: Tracking the Optimization

You can track the optimization history using the `callback` parameter. This is useful for:
- Monitoring convergence
- Stopping early if the improvement is negligible
- Plotting the convergence curve


In [ ]:
# Track convergence for Rosenbrock
history = []

def callback(xk):
    history.append(rosenbrock(xk))

x0 = np.array([-1.0, 1.0])
result = minimize(rosenbrock, x0, method='Powell', callback=callback, options={'maxiter': 10000, 'xtol': 1e-10, 'ftol': 1e-10})

plt.figure(figsize=(10, 4))
plt.plot(history, 'b-o', markersize=3, linewidth=1)
plt.xlabel('Iteration')
plt.ylabel('f(x)')
plt.title('Convergence History: Rosenbrock (Powell)')
plt.yscale('log')
plt.tight_layout()
plt.show()

print(f'Total iterations: {len(history)}')
print(f'Final f: {history[-1]:.6f}')

### What does this mean?

The convergence curve shows **rapid initial progress** (the optimizer takes large steps toward the valley) followed by **slow refinement** (zigzagging along the valley). This is typical for gradient-based methods on ill-conditioned problems.

**Practical tip:** if the convergence curve plateaus, you can stop early and save compute. The `tol` parameter controls this — set it to your desired accuracy.


## 7. Exercises

1. **Algorithm comparison.** Run L-BFGS-B, Nelder-Mead, and Powell on the Rosenbrock function from the same starting point. Compare the number of function evaluations and the final accuracy. Which is fastest?

2. **Multi-modal challenge.** Create a function with 5 local minima. Run differential_evolution with different population sizes (10, 50, 100). How does the population size affect the probability of finding the global minimum?

3. **Constrained beam.** Extend the beam design problem to include a deflection constraint: `delta = F * L^3 / (3 * E * I) <= delta_max`, where `I = w * h^3 / 12`. How does the optimal design change?

4. **Airfoil with more variables.** Add a trailing edge angle parameter to the airfoil optimization. How does the optimal design change? Does the constraint structure change?

5. **Global + local hybrid.** Run differential_evolution on the airfoil problem with a small population (20). Take the best result and refine it with SLSQP. Compare the final result to running SLSQP alone from a random start.
